In [13]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import rateslib as rl
import QuantLib as ql

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery 
from Query.FixedRateBonds.FixedRateBondStructure import FixedRateBondStructure, FixedRateBondStructureFunctionMap 
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue, FixedRateBondValueFunctionMap 

# fmt: off
import Query.FixedRateBonds.adapter  # noqa: F401
# fmt: on

from utils.ql_utils import datetime_to_ql_date, ql_date_to_datetime 

In [15]:
frb_mdp = FixedRateBondsMDP(source="USTS_PUBLICDOTCOM_WSJ_LIVE-QL")

In [32]:
ust_pricers = frb_mdp._get_multi_pricers(cusips=["ct2/ct10"], timestamp="live")
ust_pricers

{'ct2': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 7), _issue_date=datetime.date(2025, 9, 30), _maturity_date=datetime.date(2027, 9, 30), _cpn=3.5, _notional=None, _clean_price=None, _ytm=np.float64(3.578), _meta_data={'record_date': datetime.date(2025, 9, 30), 'label': 'T 3 1/2 Sep 27', 'cusip': '91282CPB1', 'oi': '2-Year', 'auction_date': datetime.date(2025, 9, 23), 'issue_date': datetime.date(2025, 9, 30), 'maturity_date': datetime.date(2027, 9, 30), 'cpn': 3.5, 'rank': 0, 'timestamp': Timestamp('2025-10-07 19:03:57+0000', tz='UTC')}),
 'ct10': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 7), _issue_date=datetime.date(2025, 8, 15), _maturity_date=datetime.date(2035, 8, 15), _cpn=4.25, _notional=None, _clean_price=None, _ytm=np.float64(4.128), _meta_data={'record_date': datetime.date(2025, 8, 15), 'label': 'T 4 1/4 Aug 35', 'cusip': '91282CNT4', 'oi': '10-Year', 'auction_date': datetime.date(2025, 8, 6), 'issue

In [34]:
# cusip = "91282CNT4"
# risk = 50_000

tenor = "Ox42/Ox410"
risk = 50_000

query = FixedRateBondQuery(cusip=tenor, structure=FixedRateBondStructure.CURVE, structure_kwargs={"bpv": risk})
pricer = frb_mdp._get_multi_pricers(cusips=[query.cusip], timestamp="live")
pkg, rws = query.resolve_package(pricer_or_curve=pricer)
vmap = query.build_value_map(pricer_or_curve=pricer, package=pkg, risk_weights=rws)

In [24]:
pkg[0].notional(), pkg[1].notional(), rws

(-300499562.72950655, 65778274.96242112, [np.float64(-1.0), np.float64(1.0)])

In [35]:
vmap.apply(FixedRateBondValue.YTM)

np.float64(44.99999999999997)

In [36]:
pricer

{'Ox42': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 7), _issue_date=datetime.date(2025, 6, 2), _maturity_date=datetime.date(2027, 5, 31), _cpn=3.875, _notional=None, _clean_price=None, _ytm=np.float64(3.611), _meta_data={'record_date': datetime.date(2025, 6, 2), 'label': 'T 3 7/8 May 27', 'cusip': '91282CNE7', 'oi': '2-Year', 'auction_date': datetime.date(2025, 5, 27), 'issue_date': datetime.date(2025, 6, 2), 'maturity_date': datetime.date(2027, 5, 31), 'cpn': 3.875, 'rank': 4, 'timestamp': Timestamp('2025-10-07 17:46:32+0000', tz='UTC')}),
 'Ox410': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 7), _issue_date=datetime.date(2024, 8, 15), _maturity_date=datetime.date(2034, 8, 15), _cpn=3.875, _notional=None, _clean_price=None, _ytm=np.float64(4.061), _meta_data={'record_date': datetime.date(2024, 8, 15), 'label': 'T 3 7/8 Aug 34', 'cusip': '91282CLF6', 'oi': '10-Year', 'auction_date': datetime.date(2024, 8, 7), 'i